In [1]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

from sklearn.model_selection import cross_val_score, GridSearchCV

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")
os.makedirs('../models', exist_ok=True)
os.makedirs('../reports', exist_ok=True)

print("Libraries imported successfully")

Libraries imported successfully


In [2]:
X_train = np.load('../data/processed/X_train.npy')
X_test  = np.load('../data/processed/X_test.npy')
y_train = np.load('../data/processed/y_train.npy')
y_test  = np.load('../data/processed/y_test.npy')

feature_names = joblib.load('../models/feature_names.pkl')

print("Data loaded successfully ")
print(f"  X_train : {X_train.shape}")
print(f"  X_test  : {X_test.shape}")
print(f"  y_train : {y_train.shape}")
print(f"  y_test  : {y_test.shape}")
print(f"\nFeatures ({len(feature_names)}): {feature_names}")

Data loaded successfully 
  X_train : (800, 12)
  X_test  : (154, 12)
  y_train : (800,)
  y_test  : (154,)

Features (12): ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'BMI_Category', 'Age_Group', 'Glucose_Level', 'Insulin_Resistance']


In [3]:
models = {

    
    'Logistic Regression': LogisticRegression(
        max_iter=1000,      
        random_state=42
    ),

    
    'Decision Tree': DecisionTreeClassifier(
        max_depth=5,        
        random_state=42
    ),

    
    'Random Forest': RandomForestClassifier(
        n_estimators=100,   
        random_state=42
    ),

    'XGBoost': XGBClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=4,
        use_label_encoder=False,
        eval_metric='logloss',
        random_state=42
    ),

    'SVM': SVC(
        kernel='rbf',       
        probability=True,   
        random_state=42
    ),

    'KNN': KNeighborsClassifier(
        n_neighbors=5       
    )
}

print(f" {len(models)} models defined and ready for training")
for name in models:
    print(f"   → {name}")

 6 models defined and ready for training
   → Logistic Regression
   → Decision Tree
   → Random Forest
   → XGBoost
   → SVM
   → KNN


In [6]:
print("Training all models on full training set...\n")

trained_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    trained_models[name] = model

    train_acc = model.score(X_train, y_train) * 100
    print(f"   {name:<25} trained | Train Accuracy: {train_acc:.2f}%")

print("\n All models trained on full training set!")

Training all models on full training set...

   Logistic Regression       trained | Train Accuracy: 81.88%
   Decision Tree             trained | Train Accuracy: 93.75%
   Random Forest             trained | Train Accuracy: 100.00%
   XGBoost                   trained | Train Accuracy: 99.50%
   SVM                       trained | Train Accuracy: 89.75%
   KNN                       trained | Train Accuracy: 87.50%

 All models trained on full training set!


In [7]:
print("  Hyperparameter Tuning — Random Forest")


rf_param_grid = {
    'n_estimators': [100, 200],        
    'max_depth': [None, 10, 20],       
    'min_samples_split': [2, 5],       
    'max_features': ['sqrt', 'log2']   
}

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    rf_param_grid,
    cv=5,
    scoring='f1',          
    n_jobs=-1,             
    verbose=0
)

rf_grid.fit(X_train, y_train)

print(f"Best RF Parameters : {rf_grid.best_params_}")
print(f"Best RF F1 Score   : {rf_grid.best_score_*100:.2f}%")

# Store the best tuned Random Forest
best_rf = rf_grid.best_estimator_
trained_models['Random Forest (Tuned)'] = best_rf

  Hyperparameter Tuning — Random Forest
Best RF Parameters : {'max_depth': None, 'max_features': 'sqrt', 'min_samples_split': 2, 'n_estimators': 100}
Best RF F1 Score   : 90.51%


In [8]:
print("  Hyperparameter Tuning — XGBoost")


xgb_param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 4, 5],
    'learning_rate': [0.05, 0.1, 0.2],
    'subsample': [0.8, 1.0]
}

xgb_grid = GridSearchCV(
    XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42),
    xgb_param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=0
)

xgb_grid.fit(X_train, y_train)

print(f"Best XGB Parameters : {xgb_grid.best_params_}")
print(f"Best XGB F1 Score   : {xgb_grid.best_score_*100:.2f}%")

best_xgb = xgb_grid.best_estimator_
trained_models['XGBoost (Tuned)'] = best_xgb

  Hyperparameter Tuning — XGBoost
Best XGB Parameters : {'learning_rate': 0.05, 'max_depth': 4, 'n_estimators': 200, 'subsample': 1.0}
Best XGB F1 Score   : 90.71%


In [9]:

print("Saving all trained models...\n")

for name, model in trained_models.items():

    filename = name.lower().replace(' ', '_').replace('(', '').replace(')', '')
    filepath = f'../models/{filename}.pkl'
    joblib.dump(model, filepath)
    print(f"   Saved: {filepath}")

print("\n All models saved to models/")

Saving all trained models...

   Saved: ../models/logistic_regression.pkl
   Saved: ../models/decision_tree.pkl
   Saved: ../models/random_forest.pkl
   Saved: ../models/xgboost.pkl
   Saved: ../models/svm.pkl
   Saved: ../models/knn.pkl
   Saved: ../models/random_forest_tuned.pkl
   Saved: ../models/xgboost_tuned.pkl

 All models saved to models/


In [ ]:
print("              MODEL TRAINING SUMMARY")

print()
print("Models trained:")
for name in trained_models:
    marker = "" if "Tuned" in name else ""
    print(f"  {marker} {name}")
print()
print("Cross-Validation Results (sorted by accuracy):")
print(f"  {'Model':<25} {'CV Accuracy':>12}")
print("  " + "-" * 40)


              MODEL TRAINING SUMMARY

Models trained:
   Logistic Regression
   Decision Tree
   Random Forest
   XGBoost
   SVM
   KNN
   Random Forest (Tuned)
   XGBoost (Tuned)

Cross-Validation Results (sorted by accuracy):
  Model                      CV Accuracy
  ----------------------------------------
  Random Forest                  90.00%
  XGBoost                        89.88%
  SVM                            87.25%
  Decision Tree                  84.88%
  KNN                            82.50%
  Logistic Regression            81.88%

Tuning done for: Random Forest & XGBoost
All models saved to: models/

  Ready for Phase 6: Evaluation & Model Selection!
